In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
df=pd.read_csv('telco.csv')
df.head()

,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,...,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,...,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,...,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [3]:
# Rename Target Variable
df = df.rename(columns={"Churn Label": "Churn"})

In [4]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [5]:
df["Churn"].value_counts()

,count
Churn,
0,5174
1,1869


To star off, we are going to drop any leakage columns, ID columns.
Clean up outliers and missing values.
Decide what to do with geo columns(possibly drop them).
Decide how to encode categorical variables.

In [6]:
# Drop Leakage Columns + ID
columns_to_drop = [
    "Quarter",
    "Customer ID",
    "Customer Status",
    "Satisfaction Score",
    "Churn Score",
    "Churn Category",
    "Churn Reason"]

df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

df.shape

(7043, 43)

In [7]:
print(f"{'Country'}: {df['Country'].unique()}\n")

Country: ['United States']



In [8]:
print(f"{'City'}: {df['City'].unique()}\n")

City: ['Los Angeles' 'Inglewood' 'Whittier' ... 'Topaz' 'Jacumba' 'Holtville']



In [9]:
# Drop Geographic Columns
geo_columns = [
    "City",# 1116 unique values, could add too much noise
    "Country", # only has one unique value
    "Zip Code",
    "Latitude",
    "Longitude",
    "State"]

df = df.drop(columns=[col for col in geo_columns if col in df.columns])

df.shape

(7043, 37)

In [10]:
# Simplify Location Information
#Create a new categorical feature [Urban_Rural] from Population
# Rural: Population <=50k / Town: 50k< Population<=200k / City: Population > 200k
df["Urban_Rural"] = pd.cut(
    df["Population"],
    bins=[-1, 50000, 200000, float("inf")],
    labels=["Rural", "Town", "City"])
df = df.drop(columns=["Population"], errors="ignore")

multi_cat_cols = ['Urban_Rural']

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)


#Drop Under 30 and Senior Citizen because Age already captures this information.
df = df.drop(columns=['Under 30','Senior Citizen'], errors="ignore")

In [11]:
df.head()

,Gender,Age,Married,Dependents,Number of Dependents,Referred a Friend,Number of Referrals,Tenure in Months,Offer,Phone Service,...,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Churn,CLTV,Urban_Rural_Town,Urban_Rural_City
0,Male,78,No,No,0,No,0,1,NaN,No,...,39.65,39.65,0.00,20,0.00,59.65,1,5433,True,False
1,Female,74,Yes,Yes,1,Yes,1,8,Offer E,Yes,...,80.65,633.30,0.00,0,390.80,1024.10,1,5302,True,False
2,Male,71,No,Yes,3,No,0,18,Offer D,Yes,...,95.45,1752.55,45.61,0,203.94,1910.88,1,3179,False,False
3,Female,78,Yes,Yes,1,Yes,1,25,Offer C,Yes,...,98.50,2514.50,13.43,0,494.00,2995.07,1,5337,False,False
4,Female,80,Yes,Yes,1,Yes,1,37,Offer C,Yes,...,76.50,2868.15,0.00,0,234.21,3102.36,1,2793,False,False


# Classify Feature Types

In [12]:
# Separate target
y = df["Churn"]

# Separate features
X = df.drop("Churn", axis=1)

X.shape, y.shape

((7043, 35), (7043,))

In [13]:
# Find categorical (object) columns
categorical_cols = X.select_dtypes(include="object").columns.tolist()
categorical_cols

['Gender',
 'Married',
 'Dependents',
 'Referred a Friend',
 'Offer',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Internet Type',
 'Online Security',
 'Online Backup',
 'Device Protection Plan',
 'Premium Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Streaming Music',
 'Unlimited Data',
 'Contract',
 'Paperless Billing',
 'Payment Method']

In [14]:
#Find numeric columns
import numpy as np

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
numeric_cols

['Age',
 'Number of Dependents',
 'Number of Referrals',
 'Tenure in Months',
 'Avg Monthly Long Distance Charges',
 'Avg Monthly GB Download',
 'Monthly Charge',
 'Total Charges',
 'Total Refunds',
 'Total Extra Data Charges',
 'Total Long Distance Charges',
 'Total Revenue',
 'CLTV']

In [15]:
#Separate binary numeric from continuous numeric
binary_cols = [col for col in numeric_cols if X[col].nunique() == 2]
continuous_cols = [col for col in numeric_cols if X[col].nunique() > 2]

binary_cols, continuous_cols

([],
 ['Age',
  'Number of Dependents',
  'Number of Referrals',
  'Tenure in Months',
  'Avg Monthly Long Distance Charges',
  'Avg Monthly GB Download',
  'Monthly Charge',
  'Total Charges',
  'Total Refunds',
  'Total Extra Data Charges',
  'Total Long Distance Charges',
  'Total Revenue',
  'CLTV'])

#Handling Missing Values

In [16]:
missing= df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

,0
Offer,3877
Internet Type,1526


In [17]:
categorical_cols = [

    "Internet Type",
    "Offer"]

# Loop through each categorical column and print unique values
for col in categorical_cols:
    print(f"{col}: {df[col].unique()}\n")

Internet Type: ['DSL' 'Fiber Optic' 'Cable' nan]

Offer: [nan 'Offer E' 'Offer D' 'Offer C' 'Offer B' 'Offer A']



In [18]:
missing_internet_type = df[df["Internet Type"].isna()]
missing_internet_type[["Internet Service", "Internet Type"]].value_counts()

,,count
Internet Service,Internet Type,


* No rows have Internet Service = Yes and Internet Type = NaN.

* All missing Internet Type rows are likely customers without internet.

* This confirms that filling with "No Internet" is the correct approach.

In [19]:
# Fill Missing Values with Meaningful Categories
df["Internet Type"] = df["Internet Type"].fillna("No Internet")
df["Offer"] = df["Offer"].fillna("No Offer")

In [20]:
# Verifying
print(df["Internet Type"].unique())
print(df["Offer"].unique())

['DSL' 'Fiber Optic' 'Cable' 'No Internet']
['No Offer' 'Offer E' 'Offer D' 'Offer C' 'Offer B' 'Offer A']


# Handling Outliers

In [21]:
for col in continuous_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
# This caps extreme values without removing rows.

# Encode Categorical Columns

In [22]:
categorical_cols = X.select_dtypes(include="object").columns.tolist()
for cols in categorical_cols:
  print(f"{cols}: {df[cols].unique()}\n")

Gender: ['Male' 'Female']

Married: ['No' 'Yes']

Dependents: ['No' 'Yes']

Referred a Friend: ['No' 'Yes']

Offer: ['No Offer' 'Offer E' 'Offer D' 'Offer C' 'Offer B' 'Offer A']

Phone Service: ['No' 'Yes']

Multiple Lines: ['No' 'Yes']

Internet Service: ['Yes' 'No']

Internet Type: ['DSL' 'Fiber Optic' 'Cable' 'No Internet']

Online Security: ['No' 'Yes']

Online Backup: ['No' 'Yes']

Device Protection Plan: ['Yes' 'No']

Premium Tech Support: ['No' 'Yes']

Streaming TV: ['No' 'Yes']

Streaming Movies: ['Yes' 'No']

Streaming Music: ['No' 'Yes']

Unlimited Data: ['No' 'Yes']

Contract: ['Month-to-Month' 'One Year' 'Two Year']

Paperless Billing: ['Yes' 'No']

Payment Method: ['Bank Withdrawal' 'Credit Card' 'Mailed Check']



## Binary Turning (0/1)

In [23]:
binary_cols = [
    "Gender",
    "Married",
    "Dependents",
    "Referred a Friend",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection Plan",
    "Premium Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Streaming Music",
    "Unlimited Data",
    "Paperless Billing"]

In [24]:
# Convert to 0/1
# Define mapping
binary_map = {
    "Yes": 1,
    "No": 0,
    "Male": 1,
    "Female": 0}

for col in binary_cols:
    df[col] = df[col].map(binary_map)

In [25]:
df[binary_cols].head()

,Gender,Married,Dependents,Referred a Friend,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data,Paperless Billing
0,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1
1,0,1,1,1,1,1,1,0,1,0,0,0,0,0,1,1
2,1,0,1,0,1,1,1,0,0,0,0,1,1,1,1,1
3,0,1,1,1,1,0,1,0,1,1,0,1,1,0,1,1
4,0,1,1,1,1,1,1,0,0,0,0,0,0,0,1,1


## Encode Multi-Category Features

In [26]:
#Columns to One-Hot Encode
multi_cat_cols = ["Offer", "Internet Type", "Contract", "Payment Method"]

In [27]:
# One-Hot Encode
X_encoded = pd.get_dummies(
    df.drop("Churn", axis=1),  # drop target
    columns=multi_cat_cols,
    drop_first=True)

y = df["Churn"]

In [28]:
# Verify
X_encoded.head()

,Gender,Age,Married,Dependents,Number of Dependents,Referred a Friend,Number of Referrals,Tenure in Months,Phone Service,Avg Monthly Long Distance Charges,...,Offer_Offer C,Offer_Offer D,Offer_Offer E,Internet Type_DSL,Internet Type_Fiber Optic,Internet Type_No Internet,Contract_One Year,Contract_Two Year,Payment Method_Credit Card,Payment Method_Mailed Check
0,1,78,0,0,0,0,0.0,1,0,0.00,...,False,False,False,True,False,False,False,False,False,False
1,0,74,1,1,0,1,1.0,8,1,48.85,...,False,False,True,False,True,False,False,False,True,False
2,1,71,0,1,0,0,0.0,18,1,11.33,...,False,True,False,False,True,False,False,False,False,False
3,0,78,1,1,0,1,1.0,25,1,19.76,...,True,False,False,False,True,False,False,False,False,False
4,0,80,1,1,0,1,1.0,37,1,6.33,...,True,False,False,False,True,False,False,False,False,False


In [29]:
# Convert Boolean Encoded Columns
bool_cols = X_encoded.select_dtypes(include='bool').columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)

In [30]:
X_encoded.head()

,Gender,Age,Married,Dependents,Number of Dependents,Referred a Friend,Number of Referrals,Tenure in Months,Phone Service,Avg Monthly Long Distance Charges,...,Offer_Offer C,Offer_Offer D,Offer_Offer E,Internet Type_DSL,Internet Type_Fiber Optic,Internet Type_No Internet,Contract_One Year,Contract_Two Year,Payment Method_Credit Card,Payment Method_Mailed Check
0,1,78,0,0,0,0,0.0,1,0,0.00,...,0,0,0,1,0,0,0,0,0,0
1,0,74,1,1,0,1,1.0,8,1,48.85,...,0,0,1,0,1,0,0,0,1,0
2,1,71,0,1,0,0,0.0,18,1,11.33,...,0,1,0,0,1,0,0,0,0,0
3,0,78,1,1,0,1,1.0,25,1,19.76,...,1,0,0,0,1,0,0,0,0,0
4,0,80,1,1,0,1,1.0,37,1,6.33,...,1,0,0,0,1,0,0,0,0,0


In [31]:
X_encoded.dtypes.value_counts()

,count
int64,36
float64,7


In [32]:
# Save the encoded version for modeling
X_encoded["Churn"] = y  # add target back to encoded dataframe
X_encoded.to_csv("telco_churn_encoded.csv", index=False)